# Lab 4：讓 Agent 研究一個真實問題（30 min）

## 目標
- 體驗 **Deep Research** 的完整流程：Planner → Searcher → Reader → Synthesizer → Critic
- 理解如何將複雜問題拆解為子問題並逐步研究
- 觀察 Critic 如何評估研究品質

## 架構
```
研究題目 → [Planner] → 子問題 → [Searcher] → 搜尋結果
         → [Reader] → 摘要 → [Synthesizer] → 報告 → [Critic] → 評估
```

In [ ]:
%%capture
!pip install langgraph langchain-openai langchain-core

In [ ]:
import os
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# --- API Key 設定 ---
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## Step 1: 定義研究題目

In [ ]:
# 研究題目
research_question = "HP EliteBook 和 Lenovo ThinkPad 在企業市場的主要差異"
print(f"研究題目：{research_question}")

## Step 2: Planner — 拆解子問題

### TODO
用 LLM 將研究題目拆解為 3 個可搜尋的子問題。

In [ ]:
def planner(question: str) -> list[str]:
    """將研究題目拆解為子問題"""
    # =============================================
    # TODO: 用 LLM 拆解子問題（約 5 行）
    # 提示：
    # 1. 寫一個 prompt 要求 LLM 將問題拆成 3 個子問題
    # 2. 要求每個子問題一行，用數字編號
    # 3. 呼叫 llm.invoke()
    # 4. 將回覆按行分割，過濾空行
    # 5. 回傳子問題列表
    #
    # prompt = f"""將以下研究題目拆解為 3 個具體的子問題，
    # 每個子問題應該可以獨立搜尋和回答。
    # 每行一個子問題，用數字編號（1. 2. 3.）。
    # 研究題目：{question}"""
    # response = llm.invoke([HumanMessage(content=___)])
    # sub_questions = [line.strip() for line in response.content.split("\n") if line.strip()]
    # return sub_questions
    # =============================================
    pass


sub_questions = planner(research_question)
# 如果 planner 還沒實作，用預設值
if not sub_questions:
    sub_questions = [
        "1. HP EliteBook 和 Lenovo ThinkPad 在硬體規格和效能上有什麼差異？",
        "2. 兩者在企業安全功能和管理工具方面有什麼不同？",
        "3. 企業採購時，兩者的價格、保固和售後服務有什麼差異？",
    ]

In [ ]:
# 印出子問題
print("Planner 拆解出的子問題：")
print("=" * 50)
for i, q in enumerate(sub_questions):
    print(f"  {q}")
print()

## Step 3: Searcher + Reader — 搜尋與摘要

Searcher 負責搜尋，Reader 負責將搜尋結果摘要成重點。

In [ ]:
# 模擬搜尋結果（實際場景中可以接 Tavily、Google Search API 等）
MOCK_SEARCH_RESULTS = {
    "硬體規格": [
        "HP EliteBook 855 G10 搭載 AMD Ryzen 7 PRO 處理器，支援 DDR5 記憶體最高 64GB。ThinkPad T14s Gen 4 採用 Intel Core i7-1365U 或 AMD Ryzen 7 PRO 7840U，最高支援 32GB LPDDR5。",
        "EliteBook 系列螢幕選項包括 Sure View 防窺螢幕，亮度高達 1000 nits。ThinkPad 則以鍵盤手感著稱，配備 TrackPoint 小紅點。",
        "兩者都支援 Thunderbolt 4，但 EliteBook 多了 HP Tamper Lock 物理安全感應器。ThinkPad 的 MIL-STD-810H 軍規測試項目通常較多。",
    ],
    "安全功能": [
        "HP Wolf Security 提供硬體層級的安全防護，包括 Sure Start（BIOS 自動修復）、Sure Click（瀏覽器隔離）、Sure Sense（AI 防毒）。",
        "Lenovo ThinkShield 安全平台包括 Self-Healing BIOS、ThinkPad PrivacyGuard 和整合的 FIDO2 認證。兩者都支援 TPM 2.0。",
        "HP Endpoint Security Controller 是獨立安全晶片，即使 OS 被攻陷也能保護。Lenovo 則提供 Pluton 安全處理器選項。",
    ],
    "價格保固": [
        "HP EliteBook 系列企業採購價約 NT$35,000-55,000，標配 3 年保固含到府維修。Lenovo ThinkPad T/X 系列約 NT$33,000-52,000，標配 3 年保固。",
        "HP 提供 HP Care Pack 延長保固最長 5 年，並有意外損壞保護（ADP）。Lenovo Premier Support 提供專屬客服熱線和優先維修。",
        "大量採購（100 台以上）時 HP 通常提供客製化服務（CTO），Lenovo 則以彈性租賃方案（DaaS）見長。",
    ],
}


def searcher(sub_question: str) -> list[str]:
    """模擬搜尋引擎，回傳相關結果"""
    # 根據關鍵字匹配模擬結果
    if any(kw in sub_question for kw in ["硬體", "規格", "效能", "hardware"]):
        return MOCK_SEARCH_RESULTS["硬體規格"]
    elif any(kw in sub_question for kw in ["安全", "管理", "security"]):
        return MOCK_SEARCH_RESULTS["安全功能"]
    elif any(kw in sub_question for kw in ["價格", "保固", "售後", "price"]):
        return MOCK_SEARCH_RESULTS["價格保固"]
    else:
        # 預設回傳所有結果的第一條
        return [v[0] for v in MOCK_SEARCH_RESULTS.values()]


# 測試 Searcher
test_results = searcher(sub_questions[0])
print(f"搜尋結果（{len(test_results)} 筆）：")
for r in test_results:
    print(f"  - {r[:60]}...")

In [ ]:
def reader(sub_question: str, search_results: list[str]) -> str:
    """Reader：將搜尋結果摘要成重點"""
    # =============================================
    # TODO: 用 LLM 摘要搜尋結果（約 3 行）
    # 提示：
    # 1. 將搜尋結果合併成字串
    # 2. 寫 prompt 要求 LLM 針對子問題摘要重點
    # 3. 回傳摘要文字
    #
    # context = "\n".join(search_results)
    # prompt = f"""根據以下搜尋結果，針對問題整理出 3-5 個重點。
    # 問題：{sub_question}
    # 搜尋結果：{context}
    # 用繁體中文回答，條列式呈現。"""
    # response = llm.invoke([HumanMessage(content=___)])
    # return response.content
    # =============================================
    pass


# 對每個子問題執行 Search + Read
sub_answers = []
for q in sub_questions:
    results = searcher(q)
    summary = reader(q, results)
    if summary:
        sub_answers.append({"question": q, "answer": summary})
    else:
        sub_answers.append({"question": q, "answer": "（Reader 尚未實作）"})

# 印出各子問題的摘要
for sa in sub_answers:
    print("=" * 50)
    print(f"子問題：{sa['question']}")
    print(f"摘要：{sa['answer']}")
    print()

## Step 4: Synthesizer + Critic — 彙整與評估

### TODO
完成 Synthesizer：將所有子問題的摘要合併成一份完整報告。

In [ ]:
def synthesizer(question: str, sub_answers: list[dict]) -> str:
    """Synthesizer：將子問題的答案合併成完整報告"""
    # =============================================
    # TODO: 用 LLM 彙整報告（約 3 行）
    # 提示：
    # 1. 將所有子問題和答案格式化成字串
    # 2. 寫 prompt 要求 LLM 整合成一份 300 字以內的研究報告
    # 3. 回傳報告
    #
    # findings = "\n\n".join([f"### {sa['question']}\n{sa['answer']}" for sa in sub_answers])
    # prompt = f"""你是研究分析師。根據以下研究發現，撰寫一份 300 字以內的研究報告。
    # 研究題目：{question}
    # 研究發現：
    # {findings}
    # 請用繁體中文撰寫，包含結論和建議。"""
    # response = llm.invoke([HumanMessage(content=___)])
    # return response.content
    # =============================================
    pass

In [ ]:
def critic(question: str, report: str) -> dict:
    """Critic：評估研究報告的品質"""
    prompt = f"""你是研究品質審查員。請評估以下研究報告：

研究題目：{question}

報告內容：
{report}

請從以下四個面向評分（1-5 分）並說明理由：
1. 完整性：是否涵蓋主要面向？
2. 準確性：資訊是否正確？
3. 實用性：對決策者是否有幫助？
4. 深度：分析是否夠深入？

最後給出整體評價：sufficient（足夠）或 insufficient（不足），並列出需要補充的方向。
用繁體中文回答。"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return {
        "evaluation": response.content,
        "is_sufficient": "sufficient" in response.content.lower() and "insufficient" not in response.content.lower(),
    }

In [ ]:
# 執行 Synthesizer
report = synthesizer(research_question, sub_answers)

if not report:
    report = "（Synthesizer 尚未實作，請先完成 TODO）"

print("=" * 60)
print("研究報告")
print("=" * 60)
print(report)
print()

# 執行 Critic
print("=" * 60)
print("Critic 評估")
print("=" * 60)
evaluation = critic(research_question, report)
print(evaluation["evaluation"])
print(f"\n整體判定：{'通過' if evaluation['is_sufficient'] else '需要補充'}")

## 觀察：Critic 覺得哪裡不夠？如果再跑一輪會補什麼？

思考以下問題：
1. Critic 指出了哪些不足之處？
2. 如果讓 Planner 根據 Critic 的回饋再拆出新的子問題，會問什麼？
3. 什麼時候應該停止研究？（迭代次數？品質分數？）
4. 模擬搜尋結果 vs 真實搜尋，品質差異會有多大？

## 延伸挑戰

把以上四個步驟用 **LangGraph** 串成完整的 Graph，加入 Conditional Edge 實現自動迭代：

```
START → Planner → Searcher → Reader → Synthesizer → Critic
                                                      ↓
                                              [sufficient?]
                                              YES → END
                                              NO  → Planner（補充研究）
```

提示：
1. State 需要加入 `iteration` 計數器（避免無限循環）
2. Critic 回傳的 `is_sufficient` 決定要不要再跑一輪
3. 設定最大迭代次數（例如 3 次）

```python
class ResearchState(TypedDict):
    question: str
    sub_questions: list[str]
    sub_answers: list[dict]
    report: str
    evaluation: str
    is_sufficient: bool
    iteration: int
```